# Hyperparameter Optimization

This notebook performs grid search to find the best hyperparameters for various regression models and saves the results to a text file.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import warnings
import os
import joblib
from itertools import product
import time

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

try:
    from lightgbm import LGBMRegressor
except ImportError:
    LGBMRegressor = None
    print("LightGBM is not installed. LightGBM model will not be available.")

try:
    from catboost import CatBoostRegressor
except ImportError:
    CatBoostRegressor = None
    print("CatBoost is not installed. CatBoost model will not be available.")

warnings.filterwarnings("ignore")
# Set global random seed for reproducibility
np.random.seed(42)
RANDOM_STATE = 42

# Create directory for output
output_dir = "final_work_project_April2025"
os.makedirs(output_dir, exist_ok=True)

print("Starting hyperparameter optimization...")
start_time = time.time()

Starting hyperparameter optimization...


In [2]:
# Load data
train_df = pd.read_csv("train_val_set6.csv")
test_df = pd.read_csv("test_set6.csv")

feature_cols = ['I_NH3', 'I_NO3', 'I_BOD5', 'I_COD', 'I_Turb', 'I_TSS', 'I_TDS', 'I_PH']
target_col = 'O_COD'

X_train_raw = train_df[feature_cols]
y_train = train_df[target_col]
X_test_raw = test_df[feature_cols]
y_test = test_df[target_col]

print(f"Training data shape: {X_train_raw.shape}")
print(f"Testing data shape: {X_test_raw.shape}")

Training data shape: (413, 8)
Testing data shape: (104, 8)


In [3]:
# Min-max normalization
def min_max_normalize(train, test):
    mins = train.min()
    maxs = train.max()
    ranges = maxs - mins
    ranges[ranges == 0] = 1
    train_norm = (train - mins) / ranges
    test_norm = (test - mins) / ranges
    return train_norm.values, test_norm.values

X_train, X_test = min_max_normalize(X_train_raw, X_test_raw)

# Display normalized data statistics
print("Normalized Training Data Statistics:")
print(pd.DataFrame(X_train, columns=feature_cols).describe())

Normalized Training Data Statistics:
            I_NH3       I_NO3      I_BOD5       I_COD      I_Turb       I_TSS  \
count  413.000000  413.000000  413.000000  413.000000  413.000000  413.000000   
mean     0.413628    0.081591    0.328579    0.413125    0.143632    0.255974   
std      0.190145    0.052434    0.164417    0.208825    0.132692    0.175856   
min      0.000000    0.000000    0.000000    0.000000    0.000000    0.000000   
25%      0.261261    0.064220    0.223386    0.273224    0.069316    0.134512   
50%      0.405405    0.082569    0.296684    0.392350    0.104757    0.195234   
75%      0.558559    0.096330    0.406632    0.561749    0.157210    0.325903   
max      1.000000    1.000000    1.000000    1.000000    1.000000    1.000000   

            I_TDS        I_PH  
count  413.000000  413.000000  
mean     0.305879    0.550621  
std      0.100923    0.191611  
min      0.000000    0.000000  
25%      0.241658    0.423077  
50%      0.307381    0.549451  
75%      

In [4]:
# Evaluation metrics
def compute_metrics(true, pred):
    mse = mean_squared_error(true, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(true, pred)
    r2 = r2_score(true, pred)
    return mse, rmse, mae, r2

In [5]:
# Model setup with fixed random seeds for all models
model_dict = {
    "Lasso Regression": Lasso(random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "SVR": SVR(),  # SVR doesn't accept random_state parameter
    "KNN Regression": KNeighborsRegressor(),
    "Random Forest": RandomForestRegressor(random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "XGBoost": XGBRegressor(random_state=RANDOM_STATE)
}

if LGBMRegressor is not None:
    model_dict["LightGBM"] = LGBMRegressor(force_col_wise=True, verbose=-1, random_state=RANDOM_STATE)
if CatBoostRegressor is not None:
    model_dict["CatBoost"] = CatBoostRegressor(verbose=False, random_state=RANDOM_STATE)

# Hyperparameter grids
param_grids = {
    "Lasso Regression": {"alpha": [0.001, 0.01, 0.1]},
    "Decision Tree": {
        "max_depth": [5, 10],
        "min_samples_split": [2, 5]
    },
    "SVR": {
        "C": [1, 10],
        "epsilon": [0.01, 0.1],
        "gamma": ["scale"]
    },
    "KNN Regression": {
        "n_neighbors": [5, 11],
        "metric": ["euclidean", "manhattan"]
    },
    "Random Forest": {
        "n_estimators": [100, 500],
        "max_depth": [5, 10]
    },
    "Gradient Boosting": {
        "n_estimators": [100],
        "learning_rate": [0.1],
        "max_depth": [5]
    },
    "XGBoost": {
        "n_estimators": [50, 100, 200],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.05, 0.1, 0.2],
        "subsample": [0.6, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "min_child_weight": [1, 3, 5],
        "gamma": [0, 0.1, 0.2]
    }
}

if LGBMRegressor is not None:
    param_grids["LightGBM"] = {
        "n_estimators": [50, 100, 200],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.05, 0.1, 0.2],
        "num_leaves": [31, 50, 70],
        "subsample": [0.6, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "min_child_samples": [20, 50, 100],
        "reg_alpha": [0, 0.1, 0.5],
        "reg_lambda": [0, 0.1, 0.5]
    }

if CatBoostRegressor is not None:
    param_grids["CatBoost"] = {
        "iterations": [50, 100, 200],
        "depth": [3, 5, 7],
        "learning_rate": [0.01, 0.05, 0.1, 0.2],
        "l2_leaf_reg": [1, 3, 5],
        "border_count": [32, 64, 128],
        "bagging_temperature": [0, 1, 2],
        "random_strength": [0, 1, 2],
        "grow_policy": ["SymmetricTree", "Depthwise", "Lossguide"]
    }

print(f"Number of models to optimize: {len(model_dict)}")
for name in model_dict.keys():
    print(f"- {name}")

Number of models to optimize: 9
- Lasso Regression
- Decision Tree
- SVR
- KNN Regression
- Random Forest
- Gradient Boosting
- XGBoost
- LightGBM
- CatBoost


In [6]:
def manual_grid_search(model, param_grid, X_train, y_train, X_val, y_val):
    best_score = float('inf')
    best_params = None
    best_model = None
    
    # Generate all combinations of parameters
    keys = param_grid.keys()
    values = param_grid.values()
    param_combinations = [dict(zip(keys, v)) for v in product(*values)]
    
    print(f"  Testing {len(param_combinations)} parameter combinations...")
    
    for params in param_combinations:
        if isinstance(model, (LGBMRegressor, CatBoostRegressor)):
            if isinstance(model, LGBMRegressor):
                model.set_params(**params)
            else:
                model = CatBoostRegressor(**params, verbose=False, random_state=RANDOM_STATE)
        else:
            model.set_params(**params)
        
        model.fit(X_train, y_train)
        val_pred = model.predict(X_val)
        score = mean_squared_error(y_val, val_pred)
        
        if score < best_score:
            best_score = score
            best_params = params
            best_model = model
    
    return best_model, best_params

In [7]:
# Use nested CV for all models with fixed random seed
cv_outer = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

final_params = {}  # Dictionary to store the final parameters used for each model

for name, model in model_dict.items():
    print(f"\n🔍 Processing {name}...")
    
    best_params = []

    for fold, (train_idx, val_idx) in enumerate(cv_outer.split(X_train, y_train), start=1):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        print(f"  🔍 Fold {fold}: Running grid search...")
        # Perform manual grid search
        best_model, best_params_fold = manual_grid_search(model, param_grids[name], X_tr, y_tr, X_val, y_val)
        best_params.append(best_params_fold)
        
        val_pred = best_model.predict(X_val)
        _, rmse, _, r2 = compute_metrics(y_val, val_pred)
        print(f"  ✅ Fold {fold} | R2: {r2:.4f} | RMSE: {rmse:.4f}")
    
    # Get the best parameters from the last fold
    final_best_params = best_params[-1]
    final_params[name] = final_best_params
    
    print(f"  📊 Best parameters for {name}:")
    for param, value in final_best_params.items():
        print(f"    - {param}: {value}")


🔍 Processing Lasso Regression...
  🔍 Fold 1: Running grid search...
  Testing 3 parameter combinations...
  ✅ Fold 1 | R2: 0.3202 | RMSE: 2.4936
  🔍 Fold 2: Running grid search...
  Testing 3 parameter combinations...
  ✅ Fold 2 | R2: 0.4120 | RMSE: 2.0385
  🔍 Fold 3: Running grid search...
  Testing 3 parameter combinations...
  ✅ Fold 3 | R2: 0.3566 | RMSE: 2.5246
  🔍 Fold 4: Running grid search...
  Testing 3 parameter combinations...
  ✅ Fold 4 | R2: 0.3676 | RMSE: 2.4929
  🔍 Fold 5: Running grid search...
  Testing 3 parameter combinations...
  ✅ Fold 5 | R2: 0.3677 | RMSE: 2.2381
  📊 Best parameters for Lasso Regression:
    - alpha: 0.001

🔍 Processing Decision Tree...
  🔍 Fold 1: Running grid search...
  Testing 4 parameter combinations...
  ✅ Fold 1 | R2: 0.8407 | RMSE: 1.2069
  🔍 Fold 2: Running grid search...
  Testing 4 parameter combinations...
  ✅ Fold 2 | R2: 0.4772 | RMSE: 1.9222
  🔍 Fold 3: Running grid search...
  Testing 4 parameter combinations...
  ✅ Fold 3 | R2: 

In [8]:
# Calculate total time
total_time = time.time() - start_time
hours, remainder = divmod(total_time, 3600)
minutes, seconds = divmod(remainder, 60)
print(f"\n⏱️ Total time for grid search: {int(hours)}h {int(minutes)}m {int(seconds)}s")

# Save final parameters to a text file in the specified folder
parameters_file = os.path.join(output_dir, "best_hyperparameters.txt")
with open(parameters_file, "w") as f:
    f.write("Best Hyperparameters for Each Model:\n\n")
    for name, params in final_params.items():
        f.write(f"{name}:\n")
        for param, value in params.items():
            f.write(f"  - {param}: {value}\n")
        f.write("\n")

print(f"\n💾 Best hyperparameters saved to {parameters_file}")


⏱️ Total time for grid search: 7h 11m 27s

💾 Best hyperparameters saved to final_work_project_April2025/best_hyperparameters.txt
